# Chapter 24: Non-Euclidean Geometry: A Historical Interlude

**Source orientation.** Perspectives on Projective Geometry, Sections 24.1-24.6, printed pages 465-482, PDF pages 487-504. The source span was used for coverage and terminology only; this notebook is an original computational lesson.

**Chapter question.** How did the search around Euclid's parallel postulate become a model-building problem?

The chapter is historical, but the mathematics is not decorative history. The central shift is this: a disputed axiom becomes testable once one can build a model in a trusted geometry. We therefore work with the unit disk as the outside observer sees it, then ask what an inside observer can measure. The notebook builds the dependency graph of Euclid's postulates, the many-parallel behavior of the Beltrami-Klein disk, a cross-ratio distance check, a Poincare contrast, and a timeline graph that records when model-building changed the logical status of non-Euclidean geometry.


## Computational Translation Guide

| Source idea | Computational object | What to inspect |
| --- | --- | --- |
| Inhabitant inside a fundamental conic | Points with Euclidean norm less than one | Equal intrinsic steps shrink in the outside picture near the boundary. |
| Euclid's first four postulates | Incidence, extension, metric circles, and right-angle congruence nodes | The fifth postulate is the branch point. |
| Parallel postulate | A line inside the disk and a point off it | Many chords through the point avoid the given chord. |
| Beltrami-Klein model | Straight Euclidean chords with distance read from boundary cross-ratio | Lines stay straight, but Euclidean angles are not reliable. |
| Poincare model | The same disk after central deformation from Klein coordinates | Angles are visually faithful, while geodesics become circular arcs. |
| Beltrami, Klein, Poincare | A model dependency timeline | Formal non-Euclidean systems become consistency models. |


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "scripts" / "ppg_inventory.py").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    possible = START / "Perspectives-on-Projective-Geometry"
    if (possible / "AGENTS.md").exists() and (possible / "scripts" / "ppg_inventory.py").exists():
        BOOK_ROOT = possible
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER_SLUG = "chapter-24-non-euclidean-geometry-a-historical-interlude"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / CHAPTER_SLUG
NOTEBOOK_PATH = BOOK_ROOT / "part-03-measurements" / CHAPTER_SLUG / "24-non-euclidean-geometry-a-historical-interlude.ipynb"
ARTIFACT_ROOT


In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.artifacts import (
    artifact_path,
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_json,
    save_table,
)
from utils.cayley_klein import klein_to_poincare, poincare_distance, poincare_to_klein
from utils.projective import cross_ratio, mobius_real

ensure_artifact_root(ARTIFACT_ROOT)
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 180})

COLORS = {
    "ink": "#1f2933",
    "muted": "#6b7280",
    "grid": "#d7dce2",
    "blue": "#2f6fbb",
    "green": "#3f7f54",
    "red": "#b4463a",
    "gold": "#b8872b",
    "purple": "#7655a6",
}

artifact_manifest = []
numeric_checks = {}


def rel(path):
    return book_relative(Path(path))


def register(path, concept):
    item = {"concept": concept, "path": rel(path), "size": int(Path(path).stat().st_size)}
    artifact_manifest.append(item)
    return path


def save_figure(fig, category, filename, concept):
    path = artifact_path(ARTIFACT_ROOT, category, filename)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return register(path, concept)


def disk_axis(ax, title=None):
    theta = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), color=COLORS["ink"], lw=1.8)
    ax.axhline(0, color=COLORS["grid"], lw=0.8)
    ax.axvline(0, color=COLORS["grid"], lw=0.8)
    ax.set_aspect("equal")
    ax.set_xlim(-1.12, 1.12)
    ax.set_ylim(-1.12, 1.12)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=12)


def disk_endpoints(point, direction):
    p = np.asarray(point, dtype=float)
    d = np.asarray(direction, dtype=float)
    d = d / np.linalg.norm(d)
    b = 2.0 * float(p @ d)
    c = float(p @ p) - 1.0
    roots = np.sort(np.roots([1.0, b, c]).real)
    return p + roots[0] * d, p + roots[1] * d


def chord_sample(a, b, samples=220, margin=1e-3):
    t = np.linspace(margin, 1 - margin, samples)
    return (1 - t)[:, None] * np.asarray(a) + t[:, None] * np.asarray(b)


def line_sample(point, direction, samples=220):
    a, b = disk_endpoints(point, direction)
    return chord_sample(a, b, samples=samples)


def klein_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    d = q - p
    acoef = float(d @ d)
    bcoef = 2.0 * float(p @ d)
    ccoef = float(p @ p) - 1.0
    t0, t1 = np.sort(np.roots([acoef, bcoef, ccoef]).real)
    cr = ((1 - t0) * t1) / ((0 - t0) * (t1 - 1))
    return 0.5 * math.log(cr), float(cr), (float(t0), float(t1))


def angle_between(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    cosang = float((u @ v) / (np.linalg.norm(u) * np.linalg.norm(v)))
    return math.acos(max(-1.0, min(1.0, cosang)))


def triangle_angles_poincare(points):
    pts = [np.asarray(p, dtype=float) for p in points]
    angles = []
    eps = 1e-5
    for i, z in enumerate(pts):
        w = poincare_to_klein(z)
        u = poincare_to_klein(pts[(i - 1) % 3]) - w
        v = poincare_to_klein(pts[(i + 1) % 3]) - w
        tu = klein_to_poincare(w + eps * u / np.linalg.norm(u)) - klein_to_poincare(w - eps * u / np.linalg.norm(u))
        tv = klein_to_poincare(w + eps * v / np.linalg.norm(v)) - klein_to_poincare(w - eps * v / np.linalg.norm(v))
        angles.append(angle_between(tu, tv))
    return np.array(angles)


def relative_image_stats(path):
    stats = image_stats(path)
    stats["path"] = rel(path)
    return stats


sample = [-1.4, -0.2, 0.75, 1.6]
image = [mobius_real(x, 1.1, -0.25, 0.22, 1.0) for x in sample]
numeric_checks["cross_ratio_error"] = float(abs(cross_ratio(*sample) - cross_ratio(*image)))


## Source-Specific Storyboard

The storyboard is saved as a check artifact before the visuals are built. Each item has an inspection target and a validation target; no artifact is a history card or a generic placeholder.


In [ ]:
storyboard = {
    "chapter goal": "Explain how non-Euclidean geometry became legitimate through explicit models of the negated parallel postulate.",
    "source span read": "Sections 24.1-24.6; printed pages 465-482; PDF pages 487-504.",
    "concept inventory": [
        "inner geometry versus an outside coordinate model",
        "Euclid's five postulates and equivalent forms of the fifth postulate",
        "Gauss, Bolyai, and Lobachevsky deriving a rich geometry from the negated fifth postulate",
        "Beltrami's consistency model and Klein's projective streamlining",
        "Beltrami-Klein disk: points inside a real conic, lines as chords, distances from boundary cross-ratio",
        "Poincare disk: the same hyperbolic plane in a conformal model",
    ],
    "library routing table": [
        {"concept": "postulate and history dependencies", "representation": "directed graphs", "library": "networkx + matplotlib", "why": "the chapter is about logical and model dependencies, not portraits"},
        {"concept": "Klein disk lines and many parallels", "representation": "unit disk chord computations", "library": "numpy + matplotlib", "why": "straight chords make the projective model inspectable"},
        {"concept": "Poincare contrast", "representation": "central deformation and interactive HTML", "library": "plotly", "why": "paired traces expose the same geodesics in two models"},
        {"concept": "distance and angle checks", "representation": "numeric invariants", "library": "numpy + local Cayley-Klein/projective helpers", "why": "checks tie drawings to the model definitions"},
    ],
    "visual sequence": [
        {"concept": "inner geometry boundary walk", "artifact": "figures/inner-geometry-boundary-walk.png", "inspection": "equal intrinsic radial steps look smaller to the outside observer", "validation": "radii stay inside the disk and Euclidean step gaps decrease"},
        {"concept": "Euclid postulate dependency graph", "artifact": "figures/euclid-postulate-dependency-graph.png", "inspection": "which statements branch from the fifth postulate", "validation": "graph contains the five postulates and fifth-postulate equivalents"},
        {"concept": "parallel postulate failure in the Klein disk", "artifact": "figures/parallel-postulate-klein-pencil.png", "inspection": "many chords through one point miss the given chord", "validation": "more than one nonintersecting chord is counted"},
        {"concept": "Beltrami-Klein distance from boundary cross-ratio", "artifact": "figures/beltrami-klein-cross-ratio-distance.png", "inspection": "distance is read from the two ideal endpoints of a chord", "validation": "cross-ratio distance is positive and symmetric"},
        {"concept": "Poincare conformal contrast", "artifact": "figures/poincare-klein-model-contrast.png", "inspection": "straightness is traded for angle fidelity", "validation": "Klein-to-Poincare round trip residual is small"},
        {"concept": "historical model-building timeline", "artifact": "figures/historical-model-building-timeline.png", "inspection": "how formal systems became model comparisons", "validation": "timeline graph has directed paths to Klein and Poincare"},
    ],
    "proof-visualization strategy": "Use dependency graphs for axiom equivalences and model chronology; use invariant trackers for distance, angle, and parallel behavior.",
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
register(storyboard_path, "source-specific storyboard")
storyboard_path


## Inner Geometry: The Boundary Is Infinitely Far Away

The opening thought experiment asks how a finite disk can feel infinite from the inside. Equal intrinsic steps along a radius have outside radii `tanh(s)`, so the Euclidean gaps get smaller while the intrinsic distance keeps increasing. The outside picture sees shrinking; the inhabitant's ruler shrinks with everything else and records no local alarm.


In [ ]:
steps = np.arange(0, 8)
intrinsic_step = 0.46
radii = np.tanh(steps * intrinsic_step)
gaps = np.diff(radii)

fig, ax = plt.subplots(figsize=(7.2, 6.0))
disk_axis(ax, "Inner geometry: equal intrinsic steps compress near the boundary")
for r in radii[1:]:
    theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(r * np.cos(theta), r * np.sin(theta), color=COLORS["grid"], lw=0.9)
ax.plot(radii, np.zeros_like(radii), color=COLORS["blue"], lw=2.2, marker="o")
for k, r in enumerate(radii):
    ax.text(r, 0.045, str(k), ha="center", va="bottom", fontsize=8)
for k in range(len(radii) - 1):
    mid = 0.5 * (radii[k] + radii[k + 1])
    ax.text(mid, -0.085, f"{gaps[k]:.3f}", ha="center", va="top", fontsize=7, color=COLORS["muted"])
ax.text(-1.04, -1.04, "labels are outside Euclidean gaps; intrinsic gaps are equal", fontsize=8, color=COLORS["muted"])
inner_walk_path = save_figure(fig, "figures", "inner-geometry-boundary-walk.png", "inner geometry boundary walk")

numeric_checks["radial_radii_inside_disk"] = bool(np.all(radii < 1))
numeric_checks["radial_gaps_strictly_decrease"] = bool(np.all(np.diff(gaps) < 0))
numeric_checks["last_radial_gap"] = float(gaps[-1])
inner_walk_path


## Euclid's Postulates as a Dependency Graph

The chapter emphasizes that the fifth postulate is unlike the first four: it became the place where equivalent statements accumulated. This graph is a proof-state scaffold. It shows which familiar facts travel with the parallel postulate once the first four postulates and the usual background assumptions are in force.


In [ ]:
G = nx.DiGraph()
postulate_nodes = {
    "E1": "E1\nsegment joins\ntwo points",
    "E2": "E2\nsegments extend",
    "E3": "E3\ncircles from\ncenter/radius",
    "E4": "E4\nright angles\ncongruent",
    "E5": "E5\nparallel\ncondition",
}
equivalent_nodes = {
    "Playfair": "Playfair\none parallel",
    "TriangleSum": "triangle\nangle sum = pi",
    "Similarity": "similar but\nnot congruent\ntriangles",
    "Rectangle": "three right\nangles imply\nfourth",
    "Pythagoras": "Pythagorean\ntheorem",
    "UnboundedArea": "triangle area\nunbounded",
}
model_nodes = {"Euclidean": "Euclidean\nmodel", "Hyperbolic": "hyperbolic\nmodel", "Elliptic": "elliptic\noption"}
for key, label in {**postulate_nodes, **equivalent_nodes, **model_nodes}.items():
    G.add_node(key, label=label)
for key in ["E1", "E2", "E3", "E4"]:
    G.add_edge(key, "Euclidean")
    G.add_edge(key, "Hyperbolic")
G.add_node("not E5", label="negated\nfifth")
G.add_node("drop infinite extent", label="drop implicit\ninfinite-line\nassumption")
G.add_edge("E5", "Playfair")
for key in ["TriangleSum", "Similarity", "Rectangle", "Pythagoras", "UnboundedArea"]:
    G.add_edge("Playfair", key)
    G.add_edge(key, "Playfair")
G.add_edge("E5", "Euclidean")
G.add_edge("not E5", "Hyperbolic")
G.add_edge("drop infinite extent", "Elliptic")

pos = {
    "E1": (-3.2, 1.5), "E2": (-3.2, 0.5), "E3": (-3.2, -0.5), "E4": (-3.2, -1.5),
    "E5": (-0.8, 1.35), "Playfair": (0.75, 1.35), "TriangleSum": (2.55, 2.0),
    "Similarity": (2.55, 1.25), "Rectangle": (2.55, 0.5), "Pythagoras": (2.55, -0.25),
    "UnboundedArea": (2.55, -1.0), "not E5": (-0.6, -1.2), "Euclidean": (4.25, 1.1),
    "Hyperbolic": (1.15, -1.55), "drop infinite extent": (-1.0, -2.35), "Elliptic": (1.2, -2.55),
}
fig, ax = plt.subplots(figsize=(10.2, 6.6))
ax.set_axis_off()
node_colors = []
for node in G.nodes:
    if node in postulate_nodes:
        node_colors.append("#dbeafe")
    elif node in equivalent_nodes:
        node_colors.append("#fef3c7")
    elif node in model_nodes:
        node_colors.append("#dcfce7")
    else:
        node_colors.append("#fee2e2")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, width=1.2, edge_color="#8b949e")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, edgecolors="#4b5563", linewidths=1.0, node_size=1850)
nx.draw_networkx_labels(G, pos, labels=nx.get_node_attributes(G, "label"), font_size=8, ax=ax)
ax.set_title("Euclid's fifth postulate as a dependency branch")
postulate_graph_path = save_figure(fig, "figures", "euclid-postulate-dependency-graph.png", "Euclid postulate dependency graph")

equiv_rows = [
    {"statement": "Playfair form", "travels_with": "fifth postulate", "hyperbolic_status": "violated: many parallels"},
    {"statement": "triangle angle sum is pi", "travels_with": "fifth postulate", "hyperbolic_status": "replaced by angle defect"},
    {"statement": "similar noncongruent triangles exist", "travels_with": "fifth postulate", "hyperbolic_status": "scale is tied to curvature"},
    {"statement": "rectangle/square-walk closure", "travels_with": "fifth postulate", "hyperbolic_status": "square walk has drift"},
    {"statement": "unbounded triangle area", "travels_with": "fifth postulate", "hyperbolic_status": "area is bounded by angle defect"},
]
postulate_table_path = save_table(equiv_rows, ARTIFACT_ROOT, "tables", "postulate-equivalence-status.csv")
register(postulate_table_path, "postulate equivalence status table")

numeric_checks["postulate_graph_nodes"] = int(G.number_of_nodes())
numeric_checks["postulate_graph_edges"] = int(G.number_of_edges())
numeric_checks["has_five_named_postulates"] = all(node in G for node in ["E1", "E2", "E3", "E4", "E5"])
postulate_graph_path


## The Parallel Postulate in the Beltrami-Klein Disk

In the Beltrami-Klein model the points are inside the unit circle and the lines are ordinary chords clipped by the circle. That makes the failure of Playfair's form visible: fix one chord `l` and a point `p` not on it. Many Euclidean lines through `p` intersect the Euclidean extension of `l`, but their intersection lies outside the active chord, so the corresponding hyperbolic lines do not meet `l`.


In [ ]:
base_y = -0.32
base_half_width = math.sqrt(1 - base_y**2)
p = np.array([0.12, 0.48])
angles = np.linspace(0.06, math.pi - 0.06, 71)
classified = []
for theta in angles:
    d = np.array([math.cos(theta), math.sin(theta)])
    t = (base_y - p[1]) / d[1]
    x_intersection = p[0] + t * d[0]
    cuts = abs(x_intersection) < base_half_width
    classified.append((theta, d, cuts, x_intersection))
non_cutting = [row for row in classified if not row[2]]
cutting = [row for row in classified if row[2]]

fig, ax = plt.subplots(figsize=(7.4, 6.2))
disk_axis(ax, "Many Klein lines through p avoid the fixed line l")
ax.plot([-base_half_width, base_half_width], [base_y, base_y], color=COLORS["ink"], lw=2.4, label="fixed hyperbolic line l")
for theta, d, cuts, _ in classified[::3]:
    pts = line_sample(p, d, samples=80)
    color = COLORS["green"] if not cuts else COLORS["grid"]
    lw = 1.5 if not cuts else 0.75
    alpha = 0.82 if not cuts else 0.5
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=lw, alpha=alpha)
ax.scatter([p[0]], [p[1]], color=COLORS["red"], s=48, zorder=5)
ax.text(p[0] + 0.04, p[1] + 0.04, "p", color=COLORS["red"], fontsize=11)
ax.text(-0.96, -0.96, f"sampled pencil: {len(non_cutting)} nonintersecting chords, {len(cutting)} cutting chords", fontsize=8, color=COLORS["muted"])
ax.legend(loc="upper left", fontsize=8)
parallel_path = save_figure(fig, "figures", "parallel-postulate-klein-pencil.png", "parallel postulate failure in the Klein disk")

numeric_checks["parallel_noncutting_count"] = int(len(non_cutting))
numeric_checks["parallel_cutting_count"] = int(len(cutting))
numeric_checks["parallel_postulate_violated_by_many"] = bool(len(non_cutting) > 1)
parallel_path


## Beltrami-Klein Distance: The Boundary Supplies the Scale

Klein's projective description is not just a picture of chords. The two ideal boundary points on the chord through `P` and `Q` supply the distance scale by a cross-ratio. This is the model-building answer to the historical question: if the Euclidean disk is trusted, a new geometry can be represented by definitions inside it.


In [ ]:
P = np.array([-0.28, -0.18])
Q = np.array([0.56, 0.32])
D = Q - P
A, B = disk_endpoints(P, D)
d_klein, cr_value, roots = klein_distance(P, Q)
d_reverse, cr_reverse, _ = klein_distance(Q, P)

fig, ax = plt.subplots(figsize=(7.2, 6.0))
disk_axis(ax, "Beltrami-Klein distance from chord endpoint cross-ratio")
chord = chord_sample(A, B)
ax.plot(chord[:, 0], chord[:, 1], color=COLORS["blue"], lw=2.2, label="Klein line")
ax.scatter([A[0], B[0]], [A[1], B[1]], color=COLORS["gold"], s=54, label="ideal endpoints A,B")
ax.scatter([P[0], Q[0]], [P[1], Q[1]], color=COLORS["red"], s=48, label="interior points P,Q")
for label, point, offset in [("A", A, (-0.06, -0.06)), ("P", P, (-0.04, 0.05)), ("Q", Q, (0.03, 0.05)), ("B", B, (0.03, -0.07))]:
    ax.text(point[0] + offset[0], point[1] + offset[1], label, fontsize=10, weight="bold")
ax.text(-1.04, -1.02, f"d_K(P,Q) = 1/2 log([A,P,Q,B]) = {d_klein:.4f}", fontsize=9, color=COLORS["ink"])
ax.text(-1.04, -0.90, "The boundary is the reference object for measurement.", fontsize=8, color=COLORS["muted"])
ax.legend(loc="upper left", fontsize=8)
klein_distance_path = save_figure(fig, "figures", "beltrami-klein-cross-ratio-distance.png", "Beltrami-Klein cross-ratio distance")

numeric_checks["klein_cross_ratio"] = float(cr_value)
numeric_checks["klein_distance_positive"] = bool(d_klein > 0)
numeric_checks["klein_distance_symmetry_error"] = float(abs(d_klein - d_reverse))
numeric_checks["klein_endpoint_norms"] = [float(np.linalg.norm(A)), float(np.linalg.norm(B))]
klein_distance_path


## Poincare Contrast: Same Geometry, Different Visual Contract

The Poincare disk is obtained from the Klein disk by a central deformation. In Klein coordinates, geodesics are straight, but Euclidean angles in the drawing are not the hyperbolic angles. In Poincare coordinates, the same geodesics bend into arcs, but the Euclidean crossing angle is the hyperbolic angle. The two models are not rival geometries; they are two coordinate contracts for the same geometry.


In [ ]:
line_specs = [
    (np.array([0.02, -0.06]), np.array([0.88, 0.34]), COLORS["blue"], "g1"),
    (np.array([0.18, 0.20]), np.array([-0.28, 0.86]), COLORS["green"], "g2"),
    (np.array([-0.20, 0.34]), np.array([0.74, -0.18]), COLORS["red"], "g3"),
]

fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.8))
for ax, title in zip(axes, ["Beltrami-Klein: geodesics are chords", "Poincare: same geodesics are conformal arcs"]):
    disk_axis(ax, title)

roundtrip_samples = []
for point, direction, color, label in line_specs:
    a, b = disk_endpoints(point, direction)
    klein_pts = chord_sample(a, b, samples=260)
    poincare_pts = np.array([klein_to_poincare(x) for x in klein_pts])
    axes[0].plot(klein_pts[:, 0], klein_pts[:, 1], color=color, lw=2, label=label)
    axes[1].plot(poincare_pts[:, 0], poincare_pts[:, 1], color=color, lw=2, label=label)
    roundtrip_samples.extend(klein_pts[::60])
axes[0].legend(loc="upper left", fontsize=8)
axes[1].legend(loc="upper left", fontsize=8)

w0 = np.array([0.23, 0.13])
d1 = np.array([0.94, 0.30]); d1 = d1 / np.linalg.norm(d1)
d2 = np.array([-0.24, 0.97]); d2 = d2 / np.linalg.norm(d2)
klein_angle = angle_between(d1, d2)
eps = 1e-4
t1 = klein_to_poincare(w0 + eps * d1) - klein_to_poincare(w0 - eps * d1)
t2 = klein_to_poincare(w0 + eps * d2) - klein_to_poincare(w0 - eps * d2)
poincare_angle = angle_between(t1, t2)
z0 = klein_to_poincare(w0)
axes[0].scatter([w0[0]], [w0[1]], color=COLORS["ink"], s=36, zorder=5)
axes[1].scatter([z0[0]], [z0[1]], color=COLORS["ink"], s=36, zorder=5)
axes[0].text(-1.04, -1.02, f"Euclidean chord angle at sample = {math.degrees(klein_angle):.1f} deg", fontsize=8, color=COLORS["muted"])
axes[1].text(-1.04, -1.02, f"Poincare drawing angle = {math.degrees(poincare_angle):.1f} deg", fontsize=8, color=COLORS["muted"])
contrast_path = save_figure(fig, "figures", "poincare-klein-model-contrast.png", "Poincare and Klein model contrast")

roundtrip = [np.linalg.norm(poincare_to_klein(klein_to_poincare(x)) - x) for x in roundtrip_samples]
numeric_checks["klein_poincare_roundtrip_max_error"] = float(max(roundtrip))
numeric_checks["klein_sample_angle_degrees"] = float(math.degrees(klein_angle))
numeric_checks["poincare_sample_angle_degrees"] = float(math.degrees(poincare_angle))
numeric_checks["model_angle_difference_degrees"] = float(abs(math.degrees(klein_angle - poincare_angle)))
contrast_path


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Beltrami-Klein chords", "Poincare arcs"))
theta = np.linspace(0, 2 * np.pi, 240)
fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", line=dict(color="black", width=2), name="absolute"), row=1, col=1)
fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", line=dict(color="black", width=2), showlegend=False), row=1, col=2)
for point, direction, color, label in line_specs:
    a, b = disk_endpoints(point, direction)
    klein_pts = chord_sample(a, b, samples=180)
    poincare_pts = np.array([klein_to_poincare(x) for x in klein_pts])
    fig.add_trace(go.Scatter(x=klein_pts[:, 0], y=klein_pts[:, 1], mode="lines", line=dict(color=color, width=3), name=label), row=1, col=1)
    fig.add_trace(go.Scatter(x=poincare_pts[:, 0], y=poincare_pts[:, 1], mode="lines", line=dict(color=color, width=3), showlegend=False), row=1, col=2)
fig.update_xaxes(range=[-1.08, 1.08], scaleanchor="y", scaleratio=1, zeroline=False)
fig.update_yaxes(range=[-1.08, 1.08], zeroline=False)
fig.update_layout(title="Same hyperbolic lines under the Klein-to-Poincare deformation", width=900, height=470, template="plotly_white")
html_path = artifact_path(ARTIFACT_ROOT, "html", "klein-poincare-model-switch.html")
fig.write_html(html_path, include_plotlyjs=True, full_html=True)
register(html_path, "interactive Klein-Poincare model switch")
html_path


## Angle Defect as a Computable Replacement for Euclidean Triangle Sum

One equivalent form of the fifth postulate is that every triangle has angle sum `pi`. In the Poincare model we can compute angles from the displayed arcs because the model is conformal. The result is not a contradiction; it is a different consistent rulebook.


In [ ]:
triangle_poincare = [np.array([-0.48, -0.18]), np.array([0.46, -0.12]), np.array([0.08, 0.58])]
angles = triangle_angles_poincare(triangle_poincare)
angle_sum = float(angles.sum())
defect = float(math.pi - angle_sum)

fig, ax = plt.subplots(figsize=(7.2, 6.0))
disk_axis(ax, "Poincare triangle: angle sum is less than pi")
for i in range(3):
    z_a = triangle_poincare[i]
    z_b = triangle_poincare[(i + 1) % 3]
    w_a = poincare_to_klein(z_a)
    w_b = poincare_to_klein(z_b)
    pts_k = chord_sample(w_a, w_b, samples=220)
    pts_p = np.array([klein_to_poincare(x) for x in pts_k])
    ax.plot(pts_p[:, 0], pts_p[:, 1], color=COLORS["blue"], lw=2)
ax.scatter([p[0] for p in triangle_poincare], [p[1] for p in triangle_poincare], color=COLORS["red"], zorder=5)
for label, point in zip(["A", "B", "C"], triangle_poincare):
    ax.text(point[0] + 0.035, point[1] + 0.035, label, fontsize=10, weight="bold")
ax.text(-1.04, -1.02, f"angle sum = {math.degrees(angle_sum):.2f} deg; defect = {math.degrees(defect):.2f} deg", fontsize=9, color=COLORS["ink"])
triangle_path = save_figure(fig, "figures", "poincare-triangle-angle-defect.png", "Poincare triangle angle defect")

numeric_checks["triangle_angle_sum_degrees"] = float(math.degrees(angle_sum))
numeric_checks["triangle_angle_defect_degrees"] = float(math.degrees(defect))
numeric_checks["triangle_defect_positive"] = bool(defect > 0)
triangle_path


## Historical Interlude as a Model-Building Timeline

This chapter's history can be read as a change in proof technology. Gauss, Bolyai, and Lobachevsky developed a rich geometry from the negated parallel postulate. Beltrami and Klein changed the logical situation by building models of it inside already trusted geometry. Poincare then supplied a conformal model that connected the subject to complex analysis and later mathematics.


In [ ]:
timeline = nx.DiGraph()
events = {
    "Euclid": {"year": -300, "label": "Euclid\naxiomatic\ngeometry", "level": 0},
    "Alhazen": {"year": 1000, "label": "Alhazen and\nlater attempts\nto prove E5", "level": 1},
    "Playfair": {"year": 1795, "label": "Playfair\none-parallel\nform", "level": 0},
    "Gauss": {"year": 1825, "label": "Gauss\nunpublished\nconviction", "level": 2},
    "Bolyai": {"year": 1832, "label": "Bolyai\nAppendix", "level": 1},
    "Lobachevsky": {"year": 1829, "label": "Lobachevsky\npublic system", "level": 3},
    "Beltrami": {"year": 1868, "label": "Beltrami\nEuclidean\nmodel", "level": 1},
    "Klein": {"year": 1872, "label": "Klein\nprojective\nmodel", "level": 2},
    "Poincare": {"year": 1882, "label": "Poincare\nconformal\ndisk", "level": 3},
    "Hilbert": {"year": 1899, "label": "Hilbert\nmodern\naxioms", "level": 0},
}
for node, attrs in events.items():
    timeline.add_node(node, **attrs)
timeline.add_edges_from([
    ("Euclid", "Alhazen"), ("Euclid", "Playfair"), ("Playfair", "Gauss"),
    ("Playfair", "Bolyai"), ("Playfair", "Lobachevsky"),
    ("Gauss", "Beltrami"), ("Bolyai", "Beltrami"), ("Lobachevsky", "Beltrami"),
    ("Beltrami", "Klein"), ("Klein", "Poincare"), ("Klein", "Hilbert"), ("Poincare", "Hilbert"),
])

fig, ax = plt.subplots(figsize=(12.0, 5.8))
ax.set_axisbelow(True)
ax.grid(axis="x", color=COLORS["grid"], lw=0.8)
for node, attrs in events.items():
    year = attrs["year"]
    level = attrs["level"]
    color = COLORS["gold"] if node in {"Gauss", "Bolyai", "Lobachevsky"} else COLORS["blue"]
    if node in {"Beltrami", "Klein", "Poincare"}:
        color = COLORS["green"]
    ax.scatter([year], [level], s=900, color=color, alpha=0.18, edgecolor=color, lw=1.5, zorder=3)
    ax.text(year, level, attrs["label"], ha="center", va="center", fontsize=8, zorder=4)
for u, v in timeline.edges:
    x0, y0 = events[u]["year"], events[u]["level"]
    x1, y1 = events[v]["year"], events[v]["level"]
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops={"arrowstyle": "->", "color": "#8b949e", "lw": 1.1, "alpha": 0.75})
ax.set_xlim(-360, 1935)
ax.set_ylim(-0.65, 3.65)
ax.set_yticks([])
ax.set_xlabel("approximate year; negative values mark BCE")
ax.set_title("Historical model-building chain for non-Euclidean geometry")
timeline_path = save_figure(fig, "figures", "historical-model-building-timeline.png", "historical model-building timeline")

numeric_checks["timeline_is_dag"] = bool(nx.is_directed_acyclic_graph(timeline))
numeric_checks["timeline_has_path_to_klein"] = bool(nx.has_path(timeline, "Euclid", "Klein"))
numeric_checks["timeline_has_path_to_poincare"] = bool(nx.has_path(timeline, "Euclid", "Poincare"))
timeline_path


## Artifact Gallery

The chapter artifacts are displayed by code below and linked here for static reading.

![Inner geometry boundary walk](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/inner-geometry-boundary-walk.png)

![Euclid postulate dependency graph](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/euclid-postulate-dependency-graph.png)

![Parallel postulate Klein pencil](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/parallel-postulate-klein-pencil.png)

![Beltrami-Klein cross-ratio distance](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/beltrami-klein-cross-ratio-distance.png)

![Poincare and Klein model contrast](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/poincare-klein-model-contrast.png)

![Poincare triangle angle defect](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/poincare-triangle-angle-defect.png)

![Historical model-building timeline](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/figures/historical-model-building-timeline.png)

Interactive artifact: [Klein-Poincare model switch](../../artifacts/chapter-24-non-euclidean-geometry-a-historical-interlude/html/klein-poincare-model-switch.html).


In [ ]:
display_order = [item["path"] for item in artifact_manifest if item["path"].endswith((".png", ".html", ".csv"))]
for item in display_order:
    display_artifact(BOOK_ROOT / item, width=820, height=470)


## Applied Lab: Move Between Models Without Changing the Geometry

Use this lab as a short exploration rather than a long exercise list.

1. Pick a point in the Klein disk and move it closer to the boundary. Convert it to Poincare coordinates and compare the radial distances.
2. Pick two Klein chords through a common point. In the Klein panel they remain straight; in the Poincare panel they bend. Compute the angle in both drawings and decide which drawing preserves the hyperbolic angle.
3. Repeat the parallel-pencil test with a different fixed chord `l`. The count of nonintersecting sampled chords changes with the sampling grid, but it should never collapse to a single line.

The point of the lab is the historical point of the chapter: once a model is explicit, philosophical pressure on the parallel postulate becomes a computational and logical question.


In [ ]:
lab_points_klein = np.array([[0.0, 0.0], [0.22, 0.0], [0.40, 0.0], [0.58, 0.0], [0.72, 0.0], [0.84, 0.0], [0.91, 0.0]])
lab_points_poincare = np.array([klein_to_poincare(p) for p in lab_points_klein])
lab_rows = []
for k, z in zip(lab_points_klein, lab_points_poincare):
    lab_rows.append({
        "klein_radius": float(np.linalg.norm(k)),
        "poincare_radius": float(np.linalg.norm(z)),
        "poincare_distance_from_origin": float(poincare_distance(np.array([0.0, 0.0]), z)),
        "inspection_target": "central deformation increases boundary resolution while distance grows",
    })
lab_table_path = save_table(lab_rows, ARTIFACT_ROOT, "tables", "model-switch-radial-lab.csv")
register(lab_table_path, "model switch radial lab table")

numeric_checks["lab_poincare_radii_increase"] = bool(np.all(np.diff([row["poincare_radius"] for row in lab_rows]) > 0))
numeric_checks["lab_distances_increase"] = bool(np.all(np.diff([row["poincare_distance_from_origin"] for row in lab_rows]) > 0))
pd.DataFrame(lab_rows)


## Takeaways

- The inner geometry of the disk can be infinite even when the outside drawing is bounded.
- Euclid's fifth postulate is equivalent to several familiar metric facts once the first four postulates and background assumptions are in place.
- Gauss, Bolyai, and Lobachevsky made the negated fifth postulate mathematically rich; Beltrami and Klein made it a model-theoretic consistency question.
- In the Beltrami-Klein model, lines are visually straight and parallel failure is easy to see; in the Poincare model, angles are visually faithful.
- The same geometry can have several useful coordinate models, and the right model depends on the feature one wants to inspect.


In [ ]:
figure_paths = [BOOK_ROOT / item["path"] for item in artifact_manifest if item["path"].endswith(".png")]
html_paths = [BOOK_ROOT / item["path"] for item in artifact_manifest if item["path"].endswith(".html")]
table_paths = [BOOK_ROOT / item["path"] for item in artifact_manifest if item["path"].endswith(".csv")]
json_paths = [BOOK_ROOT / item["path"] for item in artifact_manifest if item["path"].endswith(".json")]
all_paths = figure_paths + html_paths + table_paths + json_paths
assert_artifacts(all_paths, min_size=128)

assert numeric_checks["cross_ratio_error"] < 1e-9
assert numeric_checks["radial_radii_inside_disk"]
assert numeric_checks["radial_gaps_strictly_decrease"]
assert numeric_checks["has_five_named_postulates"]
assert numeric_checks["parallel_postulate_violated_by_many"]
assert numeric_checks["klein_distance_positive"]
assert numeric_checks["klein_distance_symmetry_error"] < 1e-12
assert numeric_checks["klein_poincare_roundtrip_max_error"] < 1e-12
assert numeric_checks["triangle_defect_positive"]
assert numeric_checks["timeline_is_dag"]
assert numeric_checks["timeline_has_path_to_klein"]
assert numeric_checks["timeline_has_path_to_poincare"]
assert numeric_checks["lab_poincare_radii_increase"]

raster_stats = [relative_image_stats(path) for path in figure_paths]
assert all(stats["pixel_std"] > 1.0 for stats in raster_stats)

visual_checks = {
    "chapter": 24,
    "all_files_exist": all(path.exists() for path in all_paths),
    "cross_ratio_error": numeric_checks["cross_ratio_error"],
    "visual_count": len(figure_paths) + len(html_paths) + len(table_paths),
    "raster_artifacts": raster_stats,
    "html_artifacts": [rel(path) for path in html_paths],
    "table_artifacts": [rel(path) for path in table_paths],
    "numeric_checks": numeric_checks,
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
register(visual_checks_path, "visual and numeric checks")

final_sanity = {
    "chapter": 24,
    "source_span": "printed pp. 465-482 / PDF pp. 487-504",
    "notebook": rel(NOTEBOOK_PATH),
    "artifacts": [item for item in artifact_manifest],
    "checks": numeric_checks,
    "visual_checks": rel(visual_checks_path),
    "notebook_executed": True,
}
final_sanity_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
register(final_sanity_path, "final sanity checks")

final_sanity
